# Betti reciprocity, demonstrated on the operators

The compliance matrix of the forcing tutorial is symmetric under $\hat A$-weighting,
$\hat A_i S_{ij}=\hat A_j S_{ji}$. That is **Betti's reciprocal theorem**. Here we derive it
from the elastic equations and check every step on the actual assembled fields — the pointwise
stress identity, the reciprocal work, the interface cancellation, and finally the compliance
form and its failure when an interface is ill-posed.


In [1]:
using Pkg; Pkg.activate(joinpath(@__DIR__, ".."))
using Melinoe, ApproxFun, Printf, LinearAlgebra
Ω = 7.2921e-5; ℓ = 2

# an all-solid two-layer body, so the tangential displacement V is a genuine field everywhere
m = build_model("solid2", [0.0..0.5, 0.5..1.0], [[1.6],[0.9]],
                [r->3.0, r->2.0], [r->1.2, r->0.8], [40,40]; R_SI = 6371e3, ρ̄_SI = 5000.0, Ω_SI = Ω)
A, _, ops, ns, jr = assemble_planet(m.layers, ℓ); Ntot = sum(ns); cumN = cumsum([0; ns])
Ω̂² = (Ω/m.ω_unit)^2
fld(f,i,x) = Fun(ops[i].S, x[f*Ntot + cumN[i]+1 : f*Ntot + cumN[i+1]])   # f = 0:U, 1:V, 2:δφ
force(mask) = A \ potential_forcing(ops, ns; ℓ, amplitude = Ω̂²/3, layer_mask = mask, jrhs = jr)

# two experiments: a wobble forcing in region 1, and in region 2
x1 = force([true,false]);  x2 = force([false,true]);


  Activating project at `~/code/ge-julia/gravito-elastic`


## 1 · The claim

Load the body two ways — force $f_i$ (a wobble in region $i$), response $u_i$. Betti says the
work of one experiment's force through the other's displacement is symmetric,

$$\int f_i\cdot u_j\,dV = \int f_j\cdot u_i\,dV,$$

and — since a region's wobble-load and moment-read are the same $r^2$-in-that-region pattern —
this is exactly $\hat A_i S_{ij}=\hat A_j S_{ji}$.


## 2 · The derivation

Each state is in equilibrium, $\nabla\!\cdot\!\sigma_i + f_i = 0$, with Hooke's law
$\sigma_i = \mathsf C\,\varepsilon_i$. The one fact that drives everything is a **pointwise**
identity:

$$\sigma_i:\varepsilon_j = \varepsilon_i:\mathsf C:\varepsilon_j = \varepsilon_j:\mathsf C:\varepsilon_i = \sigma_j:\varepsilon_i,$$

the stress of one state against the strain of the other is the same either way — **because
$\mathsf C$ is symmetric** ($C_{abcd}=C_{cdab}$, i.e. stress comes from an energy). For an
isotropic solid it is manifest: $\sigma_i:\varepsilon_j = \lambda\,\theta_i\theta_j + 2\mu\,\varepsilon_i\!:\!\varepsilon_j$
($\theta=\nabla\!\cdot\!u$), symmetric in $i,j$.

Integrate, and turn force into stress with $f_i=-\nabla\!\cdot\!\sigma_i$ and one integration by
parts:

$$\int f_i\cdot u_j\,dV = \int \sigma_i:\varepsilon_j\,dV - \oint (\sigma_i\!\cdot\!\hat n)\cdot u_j\,dS.$$

The two volume integrals ($i,j$ and $j,i$) are equal by the pointwise identity, so subtracting
leaves only the boundary:

$$\int f_i\cdot u_j - \int f_j\cdot u_i = \oint\big[(\sigma_j\!\cdot\!\hat n)\cdot u_i - (\sigma_i\!\cdot\!\hat n)\cdot u_j\big]dS.$$

On the free surface $\sigma\!\cdot\!\hat n=0$; at each interface traction and displacement are
continuous, so the two faces (opposite normals) cancel. The right side vanishes — that is Betti.
We now verify each italicised claim.


## 3 · The reciprocal work $\int f_i\cdot u_j = \int f_j\cdot u_i$

For the wobble load $\Phi_i=\tfrac{\hat\Omega^2}{3}r^2$ in region $i$, the work reduces to a
radial integral over region $i$ of state $j$'s fields (a common angular factor drops):
$\int f_i\cdot u_j \propto \int_i \rho_0\, r^3\,(2U_j + \ell(\ell+1)V_j)\,dr$.


In [2]:
W(reg, x) = (U=fld(0,reg,x); V=fld(1,reg,x); ρ=m.layers[reg].ρ₀;
             sum(Fun(r -> ρ(r)*r^3*(2U(r) + ℓ*(ℓ+1)*V(r)), Chebyshev(m.layers[reg].domain))))
W12 = W(1, x2)          # force in region 1  ·  displacement of state 2
W21 = W(2, x1)          # force in region 2  ·  displacement of state 1
@printf("∫f₁·u₂ ∝ %.8e\n∫f₂·u₁ ∝ %.8e\nrelative difference = %.1e\n", W12, W21, abs(W12-W21)/abs(W12))


∫f₁·u₂ ∝ -1.61333858e-05
∫f₂·u₁ ∝ -1.61333858e-05
relative difference = 3.1e-14


## 4 · The interface term cancels

The boundary integrand at the interface is one state's traction times the other's displacement,
$T = (\sigma_i\!\cdot\!\hat n)\cdot u_j = \sigma_{rr,i}U_j + \sigma_{r\theta,i}V_j$. Because
traction and displacement are **continuous** across the interface, $T$ is the same evaluated
from the layer below or above — so in the region-by-region sum the two faces carry opposite
normals and cancel exactly. Reconstruct the traction from the fields and check:


In [3]:
function T(side, xi, xj, r)   # state-i traction · state-j displacement at r, from one layer
    U=fld(0,side,xi); V=fld(1,side,xi); κ=m.layers[side].κ(r); μ=m.layers[side].μ(r)
    σrr = (2κ-4μ/3)*U(r) + (κ+4μ/3)*r*U'(r) + (ℓ*(ℓ+1)/3*(-3κ+2μ))*V(r)
    σrθ = μ*(r*V'(r) - V(r) + U(r))
    σrr*fld(0,side,xj)(r) + σrθ*fld(1,side,xj)(r)
end
@printf("interface T from layer 1 (below) = %.8e\n", T(1, x1, x2, 0.5))
@printf("interface T from layer 2 (above) = %.8e\n", T(2, x1, x2, 0.5))
println("equal ⇒ opposite normals cancel ⇒ no interface contribution")


interface T from layer 1 (below) = -1.02473674e-08
interface T from layer 2 (above) = -1.02473674e-08
equal ⇒ opposite normals cancel ⇒ no interface contribution


## 5 · The compliance form — and what happens when the interface is ill-posed

Because the reciprocal work is region $i$'s moment in state $j$, the theorem reads
$\hat A_i S_{ij}=\hat A_j S_{ji}$. It rests entirely on the interface cancellation of §4 — so if
an interface is *not* well-posed, it fails. A **static fluid** against a solid wall is exactly
such a case (notebook 2's Longman problem); `dahlenize` repairs it. Watch the reciprocity
residual collapse from $10^{-4}$ to machine zero:


In [4]:
mf = build_model("fluidcore", [0.0..0.5, 0.5..1.0], [[1.6],[0.9]],
                 [r->3.0, r->2.0], [r->0.0, r->0.8], [40,40]; R_SI = 6371e3, ρ̄_SI = 5000.0, Ω_SI = Ω)
for (tag, mm) in (("plain (ill-posed interface)", mf), ("dahlenized (well-posed)", dahlenize(mf)))
    c = compliances(mm; core_layer = 1, Ω_SI = Ω)
    @printf("%-28s  Ât·S₁₂ - Âf·S₂₁ = %+.1e\n", tag, c.betti)
end


plain (ill-posed interface)   Ât·S₁₂ - Âf·S₂₁ = -3.5e-04
dahlenized (well-posed)       Ât·S₁₂ - Âf·S₂₁ = +1.1e-14


## Recap

Betti reciprocity is not a coincidence of the numbers — it is Hooke's law (a symmetric
$\mathsf C$) surviving one integration by parts, with the interface terms cancelling by
continuity. Each link checks out on the fields:

| step | identity | check |
|---|---|---|
| Hooke | $\sigma_i:\varepsilon_j=\sigma_j:\varepsilon_i$ | manifest ($\lambda\theta_i\theta_j+2\mu\,\varepsilon_i\!:\!\varepsilon_j$) |
| reciprocal work | $\int f_i\cdot u_j=\int f_j\cdot u_i$ | §3, to $10^{-14}$ |
| interfaces | boundary faces cancel | §4, equal from both sides |
| compliances | $\hat A_i S_{ij}=\hat A_j S_{ji}$ | §5 — and it breaks if the interface is ill-posed |

That last line is the practical payoff: a broken reciprocity is a broken interface condition,
which is why it doubles as the correctness check on `dahlenize` and on the assembly itself.
